In [1]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import sys
print(sys.executable)

/usr/bin/python3.11


In [3]:
import sys
!{sys.executable} -m pip install -U python-woc

Defaulting to user installation because normal site-packages is not writeable


In [4]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['delsuc/spike',
            'gyyang/neurogym',
            'PapenfussLab/gridss',
            'jbytecode/LinRegOutliers',
            'nasa/CCDD',
            'arm61/pylj',
            'mothur/mothur',
            'stephenslab/mashr',
            'dswah/pyGAM',
            'lsst/daf_butler'
           ]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,004a56b528d6d1de376454a8cb0fa6701dc9ea71,delsuc_spike


In [5]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

 24%|██████████████████████▏                                                                     | 578/2395 [09:59<31:21,  1.04s/it]

Got Errors {'afde90a487c9f14697a93375d7e1fd964f0f01d8': 'Key afde90a487c9f14697a93375d7e1fd964f0f01d8 not found in /da5_fast/All.sha1c/commit_47.tch'}


 36%|█████████████████████████████████▎                                                          | 866/2395 [14:58<26:30,  1.04s/it]

Got Errors {'0db8ed378cf6560fd58a4c76f2b4582446827bea': 'Key 0db8ed378cf6560fd58a4c76f2b4582446827bea not found in /da5_fast/All.sha1c/commit_13.tch'}


 37%|█████████████████████████████████▋                                                          | 876/2395 [15:09<26:21,  1.04s/it]

...

Got Errors {'f6c14c6476c15facf2502d1bd392d4b2b4a24a01': 'Key f6c14c6476c15facf2502d1bd392d4b2b4a24a01 not found in /da5_fast/All.sha1c/commit_118.tch'}


100%|██████████████████████████████████████████████████████████████████████████████████████████▉| 2394/2395 [41:24<00:01,  1.04s/it]

Got Errors {'ffa7b9e11516e37286f5b7fca34bb2e9d05f2b8e': 'Key ffa7b9e11516e37286f5b7fca34bb2e9d05f2b8e not found in /da5_fast/All.sha1c/commit_127.tch'}


100%|███████████████████████████████████████████████████████████████████████████████████████████| 2395/2395 [41:25<00:00,  1.04s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,004a56b528d6d1de376454a8cb0fa6701dc9ea71,a5b29a89ea7ff0f7d1029c6284542647e998c5b7,[c0fc90ad8c8f892779bb4e85dca216eb41b58bac],Chiron <Chiron@localhost>,1441289801,+0200,Chiron <Chiron@localhost>,1441289801,+0200,from future print compatible with conservation...,004a56b528d6d1de376454a8cb0fa6701dc9ea71,delsuc_spike
1,00749be973f4145c4b5c085c1a5dc08ba30c74d4,abd98dd01b91d2e09e416fc9f09510bec35a500a,[6b98df47dd1e5a50032bcdb704d17dc0b6fa8552],Chiron <Chiron@localhost>,1441358966,+0200,Chiron <Chiron@localhost>,1441358966,+0200,correction on erroneous name with self\n,00749be973f4145c4b5c085c1a5dc08ba30c74d4,delsuc_spike


In [6]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '004a56b528d6d1de376454a8cb0fa6701dc9ea71', 'tree': 'a5b29a89ea7ff0f7d1029c6284542647e998c5b7', 'parent': ['c0fc90ad8c8f892779bb4e85dca216eb41b58bac'], 'author': 'Chiron <Chiron@localhost>', 'author_time': 1441289801, 'author_tz': '+0200', 'committer': 'Chiron <Chiron@localhost>', 'committer_time': 1441289801, 'committer_tz': '+0200', 'message': 'from future print compatible with conservation of the doc\n'}


In [7]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [8]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,lsst_daf_butler,004a56b528d6d1de376454a8cb0fa6701dc9ea71,Chiron <Chiron@localhost>,1441289801,from future print compatible with conservation...


In [9]:
#mode a means append, so you have all your projects in the same file
yournetid='ksaravan'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

In [4]:
import pandas as pd
df_commit_data = pd.read_csv('df_commit_data.csv')
df_commit_data.head()

,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,004a56b528d6d1de376454a8cb0fa6701dc9ea71,a5b29a89ea7ff0f7d1029c6284542647e998c5b7,['c0fc90ad8c8f892779bb4e85dca216eb41b58bac'],Chiron <Chiron@localhost>,1441289801,200,Chiron <Chiron@localhost>,1441289801,200,from future print compatible with conservation...,004a56b528d6d1de376454a8cb0fa6701dc9ea71,delsuc_spike
1,00749be973f4145c4b5c085c1a5dc08ba30c74d4,abd98dd01b91d2e09e416fc9f09510bec35a500a,['6b98df47dd1e5a50032bcdb704d17dc0b6fa8552'],Chiron <Chiron@localhost>,1441358966,200,Chiron <Chiron@localhost>,1441358966,200,correction on erroneous name with self\n,00749be973f4145c4b5c085c1a5dc08ba30c74d4,delsuc_spike
2,007fd7f2b3b9db0fa22d335a3070e2485d490d46,a59f2a8d1818cd4c2d7ab9b3bb9c6359a108a97c,['9b78b3c64bdcf4daa15c13e4d8d8a7c27a5dc3cf'],MAD <madelsuc@unistra.fr>,1620122959,200,MAD <madelsuc@unistra.fr>,1620122959,200,"improvement on documentation, and NMR Notebooks\n",007fd7f2b3b9db0fa22d335a3070e2485d490d46,delsuc_spike
3,00ca02e9ff99183b75793796c8c45b9df922f95d,dd6cfdc938eaf378171958b9fc612204a311fa26,['a543f8b58ca9869eeed5525c9f1fdecf73424ed4'],Marc-Andre' Delsuc <madelsuc@unistra.fr>,1444555855,200,Marc-Andre' Delsuc <madelsuc@unistra.fr>,1444555855,200,seems to work fine !\n,00ca02e9ff99183b75793796c8c45b9df922f95d,delsuc_spike
4,00f7f82c1a6cc30e789cd45edabcabc37723ed33,2aa2577654f785c418563ab4bccc8c54b23ccd6d,['13d0fd0a2966fb5a147fe12acfdb1dd53350734f'],MAD <madelsuc@unistra.fr>,1639069505,100,MAD <madelsuc@unistra.fr>,1639069505,100,improved spectral superimposing\n,00f7f82c1a6cc30e789cd45edabcabc37723ed33,delsuc_spike


In [31]:
# Updating the project summary correctly 
dfinf = df_commit_data[
    ['project', 'commit', 'author', 'author_time', 'message']
].copy()

dfinf = dfinf.rename(columns={'project': 'project_wocid', 'commit': 'commit_sha1', 'author_time': 'time', 'message': 'commit message'})

# Overwrite the incorrect summary file
dfinf.to_csv(
    'ksaravan_project_summary.csv',
    index=False,
    sep=';', 
    header=False
)

dfinf.head()

,project_wocid,commit_sha1,author,time,commit message
0,delsuc_spike,004a56b528d6d1de376454a8cb0fa6701dc9ea71,Chiron <Chiron@localhost>,1441289801,from future print compatible with conservation...
1,delsuc_spike,00749be973f4145c4b5c085c1a5dc08ba30c74d4,Chiron <Chiron@localhost>,1441358966,correction on erroneous name with self\n
2,delsuc_spike,007fd7f2b3b9db0fa22d335a3070e2485d490d46,MAD <madelsuc@unistra.fr>,1620122959,"improvement on documentation, and NMR Notebooks\n"
3,delsuc_spike,00ca02e9ff99183b75793796c8c45b9df922f95d,Marc-Andre' Delsuc <madelsuc@unistra.fr>,1444555855,seems to work fine !\n
4,delsuc_spike,00f7f82c1a6cc30e789cd45edabcabc37723ed33,MAD <madelsuc@unistra.fr>,1639069505,improved spectral superimposing\n


In [32]:
dfinf['project_wocid'].value_counts()

project_wocid
lsst_daf_butler             8810
mothur_mothur               4610
gyyang_neurogym             3031
papenfusslab_gridss         2382
delsuc_spike                1109
dswah_pygam                 1080
stephenslab_mashr            943
jbytecode_linregoutliers     902
arm61_pylj                   550
nasa_ccdd                    428
Name: count, dtype: int64

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

## 4. Manual GitHub Data Collection

|Project |Stars |Forks |Last Commit Date |
|---|---:|---:|---|
| delsuc/spike |21 |7 |August 10, 2026 |
| gyyang/neurogym |343 |59 |March 31, 2026 |
| PapenfussLab/gridss |285 |73 |May 21, 2025 |
| jbytecode/LinRegOutliers |53 |6 |September 4, 2026 |
| nasa/CCDD |93 |33 |June 30, 2026 |
| arm61/pylj |31 |22 |September 26, 2026 |
| mothur/mothur |281 |111 |August 21, 2026 |
| stephenslab/mashr |101 |21 |November 7, 2024 |
| dswah/pyGAM |1017 |292 |January 22, 2026 |
| lsst/daf_butler |16 |20 |September 27, 2026 |

## 5th Task

In [33]:
print(type(df_commit_data))
print(df_commit_data.columns.tolist())

<class 'pandas.DataFrame'>
['commit', 'tree', 'parent', 'author', 'author_time', 'author_tz', 'committer', 'committer_time', 'committer_tz', 'message', 'sha1', 'project']


In [34]:
# Using WoC API for getting the information

df_commit_data = pd.read_csv('df_commit_data.csv')

statistics = df_commit_data.groupby('project').agg({'commit': 'nunique', 'author': 'nunique', 'author_time': ['min', 'max']})

statistics

commit  author author_time            
                         nunique nunique         min         max
project                                                         
arm61_pylj                   550      17  1517503095  1733104946
delsuc_spike                1109      33  1416251831  1762186964
dswah_pygam                 1080      80  1484798062  1760626177
gyyang_neurogym             3031      39  1550764500  1761577669
jbytecode_linregoutliers     902      10  1598126055  1761947018
lsst_daf_butler             8810      86  1412152689  1762283255
mothur_mothur               4610      50  1232041010  1768327073
nasa_ccdd                    428      13  1484077520  1734639707
papenfusslab_gridss         2382      43  1377588244  1755694478
stephenslab_mashr            943      15  1481984097  1730994282

## Final Table

|Project |Commits |Authors |Min time|Max Time |Stars |Forks |Last Commit Date |
|---|---:|---:|---:|---:|---:|---:|---|
| delsuc/spike |1109 |33 |1416251831 |1762186964 |21 |7 |August 10, 2026 |
| gyyang/neurogym |3031	|39 |1550764500 |1761577669 |343 |59 |March 31, 2026 |
| PapenfussLab/gridss |2382	|43 |1377588244 |1755694478 |285 |73 |May 21, 2025 |
| jbytecode/LinRegOutliers |902 |10 |1598126055 |1761947018 |53 |6 |September 4, 2026 |
| nasa/CCDD |428 |13 |1484077520 |1734639707 |93 |33 |June 30, 2026 |
| arm61/pylj |550 |17 |1517503095 |1733104946 |31 |22 |September 26, 2026 |
| mothur/mothur |4610 |50 |1232041010 |1768327073 |281 |111 |August 21, 2026 |
| stephenslab/mashr |943 |15 |1481984097 |1730994282 |101 |21 |November 7, 2024 |
| dswah/pyGAM |1080 |80 |1484798062 |1760626177 |1017 |292 |January 22, 2026 |
| lsst/daf_butler |8810	|86 |1412152689 |1762283255 |16 |20 |September 27, 2026 |


## Part 2


In [2]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(
    'ksaravan_project_summary.csv',
    sep=';',
    header=None,
    names=['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']
)

df.head()

,project_wocid,commit_sha1,author,time,commit message
0,delsuc_spike,004a56b528d6d1de376454a8cb0fa6701dc9ea71,Chiron <Chiron@localhost>,1441289801,from future print compatible with conservation...
1,delsuc_spike,00749be973f4145c4b5c085c1a5dc08ba30c74d4,Chiron <Chiron@localhost>,1441358966,correction on erroneous name with self\n
2,delsuc_spike,007fd7f2b3b9db0fa22d335a3070e2485d490d46,MAD <madelsuc@unistra.fr>,1620122959,"improvement on documentation, and NMR Notebooks\n"
3,delsuc_spike,00ca02e9ff99183b75793796c8c45b9df922f95d,Marc-Andre' Delsuc <madelsuc@unistra.fr>,1444555855,seems to work fine !\n
4,delsuc_spike,00f7f82c1a6cc30e789cd45edabcabc37723ed33,MAD <madelsuc@unistra.fr>,1639069505,improved spectral superimposing\n
